# Pre-Training Model

This notebook provides an overview of the process. The code cells will later be collected and moved into the training_model.py file.

## Pipeline Definition  
**1.** Step 1: Load and Basic Cleaning of Data  
- load_data()  
- basic_cleaning()  
  - clean_movies()  
  - clean_ratings()  
  - clean_tags()  
  - clean_data()  

**2.** Step 2: Split Data by User  
- split_data_by_user()  

**3.** Step 3: Build Train-Only User Processing  
- build_train_user_tables()  
- apply_cutoff()  
- build_user_features()  
- fit_preprocessing()  

**4.** Step 4: Model Selection  
- train_user_baseline_model()  
- train_clustered_user_model()  
  - fit_user_clustering()  

**5.** Step 5: Transform Validation/Test Users and Evaluate  
- transform_validation_test_users()  
- evaluate_on_val()

## Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re 
from sklearn.cluster import KMeans

## Step 1: Load and Basic Cleaning of Data

### Load data

In [2]:
data_dir = "../data"

def load_data(data_dir):
    movies = pd.read_csv(f"{data_dir}/movies.csv")
    ratings = pd.read_csv(f"{data_dir}/ratings.csv")
    tags = pd.read_csv(f"{data_dir}/tags.csv")
    return movies, ratings, tags

movies, ratings, tags = load_data(data_dir)


### Basic cleaning

#### clean_movies (movie metadata)

In [3]:
def clean_movies(movies):
    movies = movies.copy()
    movies = movies.drop_duplicates(subset=["movieId"])
    movies["title"] = movies["title"].astype(str).str.strip()
    movies["genres"] = movies["genres"].fillna("").astype(str).str.strip()
    return movies

#### clean_ratings (user-item interactions)

In [4]:
def clean_ratings(ratings):
    ratings = ratings.copy()
    ratings = ratings.drop_duplicates()
    ratings = ratings.dropna(subset=["userId", "movieId", "rating"])
    return ratings

#### clean_tags (user-generated tag data)

In [5]:
def clean_tags(tags):
    tags = tags.copy()
    tags = tags.dropna(subset=["userId", "movieId", "tag"])

    tags["tag"] = (
        tags["tag"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("-", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

    tags = tags[~tags["tag"].isin(["", "nan", "none", "null"])]
    tags = tags.drop_duplicates(subset=["userId", "movieId", "tag"])
    return tags


#### clean_data (return cleaned source tables)

In [6]:
def clean_data(movies, ratings, tags):
    movies = clean_movies(movies)
    ratings = clean_ratings(ratings)
    tags = clean_tags(tags)
    return movies, ratings, tags

## Step 2: Split Data by User (train/val/test)

In [7]:
def split_data_by_user(movies, ratings, tags, train_size=0.7, val_size=0.15, test_size=0.15):
    if not np.isclose(train_size + val_size + test_size, 1.0):
        raise ValueError("train_size, val_size, and test_size must sum to 1.0")

    user_ids = ratings["userId"].dropna().unique()
    shuffled_user_ids = np.random.permutation(user_ids)

    n_users = len(shuffled_user_ids)
    n_train = int(n_users * train_size)
    n_val = int(n_users * val_size)

    train_user_ids = shuffled_user_ids[:n_train]
    val_user_ids = shuffled_user_ids[n_train:n_train + n_val]
    test_user_ids = shuffled_user_ids[n_train + n_val:]

    ratings_train = ratings[ratings["userId"].isin(train_user_ids)].copy()
    ratings_val = ratings[ratings["userId"].isin(val_user_ids)].copy()
    ratings_test = ratings[ratings["userId"].isin(test_user_ids)].copy()

    tags_train = tags[tags["userId"].isin(train_user_ids)].copy()
    tags_val = tags[tags["userId"].isin(val_user_ids)].copy()
    tags_test = tags[tags["userId"].isin(test_user_ids)].copy()

    # The same movie may appear in multiple splits, which is expected in a user-based setup.
    train_movie_ids = pd.Index(ratings_train["movieId"]).union(tags_train["movieId"])
    val_movie_ids = pd.Index(ratings_val["movieId"]).union(tags_val["movieId"])
    test_movie_ids = pd.Index(ratings_test["movieId"]).union(tags_test["movieId"])

    movies_train = movies[movies["movieId"].isin(train_movie_ids)].copy()
    movies_val = movies[movies["movieId"].isin(val_movie_ids)].copy()
    movies_test = movies[movies["movieId"].isin(test_movie_ids)].copy()

    return {
        "train": (movies_train, ratings_train, tags_train),
        "val": (movies_val, ratings_val, tags_val),
        "test": (movies_test, ratings_test, tags_test),
    }


splits = split_data_by_user(movies, ratings, tags)
(movies_train, ratings_train, tags_train) = splits["train"]
(movies_val, ratings_val, tags_val) = splits["val"]
(movies_test, ratings_test, tags_test) = splits["test"]

## Step 3: Build Train-Only User Processing

### Build train interaction table

Since ratings.csv contains the largest number of rows, I use it as the base table before moving on to broader user feature engineering.

In [9]:
def build_interaction_table(ratings, movies, tags):
    ratings_prepared = ratings.drop(columns=["timestamp"], errors="ignore").copy()
    tags_prepared = tags.copy()
    tags_prepared["tag"] = tags_prepared["tag"].fillna("").astype(str).str.strip()
    tags_prepared = tags_prepared[tags_prepared["tag"] != ""]

    interaction_table = ratings_prepared.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    tags_agg = (
        tags_prepared.groupby(["userId", "movieId"])["tag"]
        .apply(lambda x: " ".join(x))
        .reset_index(name="tag_text")
    )

    interaction_table = interaction_table.merge(
        tags_agg,
        on=["userId", "movieId"],
        how="left"
    )

    interaction_table["tag_text"] = interaction_table["tag_text"].fillna("")
    return interaction_table


train_interactions = build_interaction_table(ratings_train, movies_train, tags_train)
val_interactions = build_interaction_table(ratings_val, movies_val, tags_val)
test_interactions = build_interaction_table(ratings_test, movies_test, tags_test)

train_interactions[train_interactions["tag_text"] != ""].head(10)

,userId,movieId,rating,title,genres,tag_text
2255,37,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,Kevin Spacey Morgan Freeman powerful ending tw...
2256,37,165,4.0,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,action
2257,37,293,5.0,Léon: The Professional (a.k.a. The Professiona...,Action|Crime|Drama|Thriller,Gary Oldman great acting Jean Reno Natalie Por...
2259,37,480,4.0,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,classic Steven Spielberg
2260,37,527,5.0,Schindler's List (1993),Drama|War,acting John Williams moving
2261,37,541,4.5,Blade Runner (1982),Action|Sci-Fi|Thriller,classic dystopia sci-fi
2262,37,593,4.5,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,acting Anthony Hopkins
2263,37,1036,5.0,Die Hard (1988),Action|Crime|Thriller,action Alan Rickman classic tense
2264,37,1127,4.5,"Abyss, The (1989)",Action|Adventure|Sci-Fi|Thriller,acting Ed Harris James Cameron
2265,37,1196,4.5,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,classic John Williams


### Apply user and item cutoff

### Build user features

#### Add optional movie metadata features

### Scale user features

## Step 4: Model Selection

### User-based baseline model

### More advanced clustered user model

#### User clustering

## Step 5: Transform Validation/Test Users and Evaluate